In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "What is the best way to reduce fatigue in Long COVID?"

# Fatigue Treatments in Long COVID: A Community Evidence Analysis

**Abstract:** Among 585 users in r/covidlonghaulers who discussed fatigue, tiredness, or exhaustion during March-April 2026, 386 also filed treatment reports covering 35+ therapies. Magnesium-based supplements and CoQ10 (coenzyme Q10, a mitochondrial support supplement) emerged as the top-performing treatments with positive rates exceeding 90%, while creatine and B12 showed strong results for energy-specific complaints. SSRIs (selective serotonin reuptake inhibitors) performed notably poorly among fatigue reporters, with only 30% positive sentiment -- well below the community baseline of 68%. LDN (low dose naltrexone), the community's most-discussed treatment overall, remained effective but showed reduced efficacy specifically in fatigue-focused posts compared to its general performance. These findings suggest that mitochondrial and nutritional support strategies outperform neuropsychiatric approaches for Long COVID fatigue, though sample sizes warrant caution for several promising treatments.


## Data Exploration

This analysis draws from the r/covidlonghaulers subreddit, one of the largest online communities for Long COVID patients. We identify fatigue reporters as users whose posts mention "fatigue," "tired," or "exhaustion" and analyze their treatment experiences.


In [ ]:

# Define fatigue cohort
fatigue_users = pd.read_sql(
    """SELECT DISTINCT user_id FROM posts
    WHERE body_text LIKE '%fatigue%'
       OR body_text LIKE '%tired%'
       OR body_text LIKE '%exhausti%'""", conn)
fatigue_ids = set(fatigue_users['user_id'])
n_fatigue = len(fatigue_ids)

# Date range
date_range = pd.read_sql("SELECT MIN(post_date) as mn, MAX(post_date) as mx FROM posts", conn)
from datetime import datetime
d_min = datetime.fromtimestamp(date_range['mn'].iloc[0]).strftime('%Y-%m-%d')
d_max = datetime.fromtimestamp(date_range['mx'].iloc[0]).strftime('%Y-%m-%d')

# Fatigue users with treatment reports
fatigue_reporters = pd.read_sql(
    "SELECT DISTINCT tr.user_id FROM treatment_reports tr WHERE tr.user_id IN ({})".format(
        ','.join("'" + u + "'" for u in fatigue_ids)), conn)
n_reporters = len(fatigue_reporters)

display(HTML(f"""
<div style="background:#f0f7ff; padding:15px; border-radius:8px; margin:10px 0;">
<h3 style="margin-top:0;">Dataset Overview</h3>
<table style="font-size:14px;">
<tr><td><b>Data covers:</b></td><td>{d_min} to {d_max} (~1 month)</td></tr>
<tr><td><b>Total community users:</b></td><td>2,827</td></tr>
<tr><td><b>Users mentioning fatigue:</b></td><td>{n_fatigue:,} ({n_fatigue/2827*100:.0f}% of community)</td></tr>
<tr><td><b>Fatigue users with treatment reports:</b></td><td>{n_reporters:,}</td></tr>
<tr><td><b>Total treatment reports:</b></td><td>6,815</td></tr>
</table>
</div>
"""))


Fatigue is the most commonly discussed symptom in this community -- over 20% of all users mention it explicitly. The 386 fatigue reporters who also have treatment data give us sufficient sample sizes for the top 15-20 treatments.

**Methodology note:** We use two cohort definitions throughout this analysis. The *broad cohort* includes all treatment reports from users who ever mentioned fatigue in any post. The *narrow cohort* restricts to treatment reports extracted from posts that themselves discuss fatigue or energy. The narrow cohort better captures fatigue-specific treatment experiences but has smaller sample sizes.

**Filtering applied:** Generic terms (supplements, medication, etc.) are excluded as non-actionable. Vaccines are excluded as causal-context contamination -- in this community, vaccines are discussed as a perceived cause of Long COVID, not as a treatment. Duplicates are merged where appropriate (pepcid/famotidine, tirzepatide/zepbound).


## Baseline: The Treatment Landscape for Fatigue Reporters

Before evaluating individual treatments, we need to understand the overall picture. How does the fatigue cohort compare to the broader community in treatment outcomes?


In [ ]:

import math

# Causal-context and generic exclusions
EXCLUDE = {
    'supplements','medication','treatment','therapy','drug','drugs','vitamin',
    'prescription','pill','pills','dosage','dose','antihistamines','antibiotics',
    'covid vaccine','flu vaccine','mmr vaccine','moderna vaccine',
    'mrna covid-19 vaccine','pfizer vaccine','vaccine','vaccine injection',
    'pfizer','booster','h1 antihistamine','h2 antihistamine'
}

MERGE_MAP = {
    'magnesium glycinate': 'magnesium',
    'pepcid': 'famotidine',
    'zepbound': 'tirzepatide',
}

# User-level aggregation for fatigue cohort (broad)
all_reports = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment, tr.signal_strength
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

all_reports['drug'] = all_reports['drug'].replace(MERGE_MAP)
all_reports = all_reports[~all_reports['drug'].isin(EXCLUDE)]
all_reports['score'] = all_reports['sentiment'].map(SENTIMENT_SCORE)

# User-level: average score per user per drug
user_level = all_reports.groupby(['user_id','drug']).agg(
    avg_score=('score','mean'),
    n_reports=('score','count')
).reset_index()
user_level['outcome'] = user_level['avg_score'].apply(classify_outcome)

# Fatigue vs non-fatigue cohorts
user_level['is_fatigue'] = user_level['user_id'].isin(fatigue_ids)
fatigue_ul = user_level[user_level['is_fatigue']].copy()
nonfatigue_ul = user_level[~user_level['is_fatigue']].copy()

# Overall positive rates
fat_pos = (fatigue_ul['outcome']=='positive').sum()
fat_total = len(fatigue_ul)
fat_rate = fat_pos / fat_total

nf_pos = (nonfatigue_ul['outcome']=='positive').sum()
nf_total = len(nonfatigue_ul)
nf_rate = nf_pos / nf_total

# Fisher's exact test
table = [[fat_pos, fat_total - fat_pos], [nf_pos, nf_total - nf_pos]]
odds_ratio, p_fisher = fisher_exact(table)

# Cohen's h
cohens_h = 2 * (math.asin(math.sqrt(fat_rate)) - math.asin(math.sqrt(nf_rate)))

display(HTML(f"""
<div style="background:#f9f9f9; padding:15px; border-radius:8px; margin:10px 0;">
<h3 style="margin-top:0;">Fatigue Cohort vs. Rest of Community</h3>
<table style="font-size:14px;">
<tr><td><b>Fatigue cohort positive rate:</b></td><td>{fat_rate:.1%} ({fat_pos:,} of {fat_total:,} user-drug pairs)</td></tr>
<tr><td><b>Non-fatigue cohort positive rate:</b></td><td>{nf_rate:.1%} ({nf_pos:,} of {nf_total:,} user-drug pairs)</td></tr>
<tr><td><b>Fisher's exact p-value:</b></td><td>{p_fisher:.4f}</td></tr>
<tr><td><b>Cohen's h:</b></td><td>{cohens_h:.3f}</td></tr>
</table>
<p style="margin-top:10px;"><b>Plain language:</b> Fatigue reporters have {'similar' if abs(cohens_h) < 0.2 else 'different'} overall treatment outcomes compared to the rest of the community (Cohen's h = {cohens_h:.3f}, {'negligible' if abs(cohens_h) < 0.2 else 'small'} effect). {'This means fatigue-specific treatment effects are more interesting than overall cohort differences.' if abs(cohens_h) < 0.2 else ''}</p>
</div>
"""))


## Core Analysis: Ranking Fatigue Treatments

The fatigue cohort's overall treatment response is similar to the broader community. The interesting question is which *specific* treatments work best for fatigue. We rank treatments by Wilson score lower bound -- a conservative estimate that penalizes small sample sizes, preventing a treatment tried by 3 people (all positive) from outranking one tried by 50 people with 80% positive.


In [ ]:

# Treatment rankings for fatigue cohort
drug_stats = fatigue_ul.groupby('drug').agg(
    users=('user_id','nunique'),
    positive=('outcome', lambda x: (x=='positive').sum()),
    negative=('outcome', lambda x: (x=='negative').sum()),
    mixed_neutral=('outcome', lambda x: (x.isin(['mixed/neutral'])).sum()),
    avg_score=('avg_score','mean')
).reset_index()

drug_stats = drug_stats[drug_stats['users'] >= 10].copy()
drug_stats['total'] = drug_stats['positive'] + drug_stats['negative'] + drug_stats['mixed_neutral']
drug_stats['pos_rate'] = drug_stats['positive'] / drug_stats['total']
drug_stats['neg_rate'] = drug_stats['negative'] / drug_stats['total']

# Wilson CI
drug_stats['wilson_lo'] = drug_stats.apply(lambda r: wilson_ci(r['positive'], r['total'])[0], axis=1)
drug_stats['wilson_hi'] = drug_stats.apply(lambda r: wilson_ci(r['positive'], r['total'])[1], axis=1)

# Baseline positive rate for NNT
baseline_rate = fat_rate

# NNT
drug_stats['nnt_val'] = drug_stats['pos_rate'].apply(lambda r: nnt(r, baseline_rate))

drug_stats = drug_stats.sort_values('wilson_lo', ascending=False).reset_index(drop=True)

# Display top 20
display_df = drug_stats.head(20).copy()
display_df['Rank'] = range(1, len(display_df)+1)
display_df['Positive Rate'] = display_df['pos_rate'].apply(lambda x: f"{x:.0%}")
display_df['95% CI'] = display_df.apply(lambda r: f"[{r['wilson_lo']:.0%}, {r['wilson_hi']:.0%}]", axis=1)
display_df['NNT'] = display_df['nnt_val'].apply(lambda x: f"{x:.1f}" if x is not None else "N/A")

styled = display_df[['Rank','drug','users','positive','negative','mixed_neutral','Positive Rate','95% CI','NNT']].rename(
    columns={'drug':'Treatment','users':'Users','positive':'Pos','negative':'Neg','mixed_neutral':'Mix/Neut'}
).style.set_properties(**{'text-align':'center'}).set_table_styles(
    [{'selector':'th','props':[('text-align','center'),('font-weight','bold')]}]
).hide(axis='index')
display(styled)


The table above shows user-level outcomes: each user's reports for a given treatment are averaged into a single score, then classified as positive (>0.7), negative (<-0.3), or mixed/neutral. This prevents prolific posters from dominating the results.

**NNT (Number Needed to Treat)** tells you how many patients would need to try a treatment for one additional person to report benefit beyond the baseline rate. Lower is better -- an NNT of 3 means roughly 1 in 3 additional patients benefits.


In [ ]:

# Forest plot: Top 15 treatments by Wilson score
plot_df = drug_stats.head(15).sort_values('wilson_lo', ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, 8))
y_pos = range(len(plot_df))

# Points and CIs
ax.errorbar(
    plot_df['pos_rate'].values, y_pos,
    xerr=[plot_df['pos_rate'].values - plot_df['wilson_lo'].values,
          plot_df['wilson_hi'].values - plot_df['pos_rate'].values],
    fmt='o', color='#2ecc71', ecolor='#95a5a6', elinewidth=2, capsize=4, markersize=8
)

# Baseline reference line
ax.axvline(x=baseline_rate, color='#e74c3c', linestyle='--', alpha=0.7, label=f'Cohort baseline ({baseline_rate:.0%})')

ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"{r['drug']} (n={r['users']})" for _, r in plot_df.iterrows()], fontsize=10)
ax.set_xlabel('Positive Outcome Rate', fontsize=12)
ax.set_title('Top 15 Fatigue Treatments: Positive Rate with 95% Wilson CI', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0.3, 1.05)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


**Key takeaway:** Magnesium leads the ranking with a 90%+ positive rate and tight confidence intervals, followed by B12, creatine, and B vitamins. These are all nutritional/mitochondrial support treatments. The red dashed line marks the cohort baseline positive rate. Treatments whose entire confidence interval falls above the baseline are reliably outperforming average community expectations.

Note that LDN (low dose naltrexone), despite being the most-discussed treatment by far (93 users), ranks in the middle tier -- its large sample size gives us high confidence that its ~65% positive rate is real, but it does not dominate the fatigue-specific rankings.


In [ ]:

# Diverging bar chart: Sentiment breakdown for top 15
div_df = drug_stats.head(15).sort_values('pos_rate', ascending=True).copy()
div_df['neg_pct'] = div_df['neg_rate'] * 100
div_df['mix_pct'] = div_df['mixed_neutral'] / div_df['total'] * 100
div_df['pos_pct'] = div_df['pos_rate'] * 100

fig, ax = plt.subplots(figsize=(11, 7))
y = range(len(div_df))

# Mixed innermost (from 0 leftward), Negative outermost
ax.barh(list(y), -div_df['mix_pct'].values, left=0, color='#95a5a6', height=0.6, label='Mixed/Neutral')
ax.barh(list(y), -div_df['neg_pct'].values, left=-div_df['mix_pct'].values, color='#e74c3c', height=0.6, label='Negative')
ax.barh(list(y), div_df['pos_pct'].values, left=0, color='#2ecc71', height=0.6, label='Positive')

ax.set_yticks(list(y))
ax.set_yticklabels([f"{r['drug']} (n={r['users']})" for _, r in div_df.iterrows()], fontsize=10)
ax.set_xlabel('Percentage of User-Level Outcomes', fontsize=11)
ax.set_title('Fatigue Treatment Outcomes: Full Sentiment Breakdown', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)

# X-axis labels
ticks = ax.get_xticks()
ax.set_xticklabels([f"{abs(int(t))}%" for t in ticks])

plt.tight_layout()
plt.show()


**Key takeaway:** Magnesium and B vitamins show almost no negative reports -- their risk profile is extremely favorable. SSRIs stand out as the only top-discussed treatment with more negative than positive outcomes among fatigue reporters. Nicotine and famotidine show substantial mixed/neutral segments, suggesting inconsistent experiences.


## Statistical Validation: Do the Top Treatments Beat Baseline?

Ranking treatments by positive rate is informative but insufficient. We need to test whether each treatment's positive rate significantly exceeds what we would expect by chance (50% baseline, since reports could be positive or negative).


In [ ]:

# Binomial tests for top treatments
results = []
for _, row in drug_stats.head(20).iterrows():
    bt = binomtest(int(row['positive']), int(row['total']), p=0.5, alternative='greater')
    h = 2 * (math.asin(math.sqrt(row['pos_rate'])) - math.asin(math.sqrt(0.5)))
    results.append({
        'Treatment': row['drug'],
        'n': int(row['users']),
        'Positive': int(row['positive']),
        'Rate': row['pos_rate'],
        'p-value': bt.pvalue,
        "Cohen's h": h,
        'Significant': bt.pvalue < 0.05
    })

res_df = pd.DataFrame(results)

# Sensitivity check: strong signals only
strong_reports = all_reports[all_reports['signal_strength']=='strong']
strong_user_ids = set(strong_reports['user_id'])
strong_fatigue_ul = fatigue_ul[fatigue_ul['user_id'].isin(strong_user_ids)]
strong_top = []
for drug in res_df.head(5)['Treatment']:
    sub = strong_fatigue_ul[strong_fatigue_ul['drug']==drug]
    if len(sub) >= 5:
        pos_count = (sub['outcome']=='positive').sum()
        tot_count = len(sub)
        strong_top.append({'drug': drug, 'n': tot_count, 'pos_rate': pos_count/tot_count})

# Format display
display_res = res_df.copy()
display_res['Rate'] = display_res['Rate'].apply(lambda x: f"{x:.0%}")
display_res['p-value'] = display_res['p-value'].apply(lambda x: f"{x:.4f}" if x >= 0.001 else f"{x:.2e}")
display_res["Cohen's h"] = display_res["Cohen's h"].apply(lambda x: f"{x:.2f}")
display_res['Sig.'] = display_res['Significant'].apply(lambda x: 'Yes' if x else 'No')

styled = display_res[['Treatment','n','Positive','Rate','p-value',"Cohen's h",'Sig.']].style.set_properties(
    **{'text-align':'center'}
).set_table_styles(
    [{'selector':'th','props':[('text-align','center'),('font-weight','bold')]}]
).apply(lambda row: ['background-color: #d4edda' if row['Sig.']=='Yes' else 'background-color: #f8d7da' for _ in row], axis=1
).hide(axis='index')
display(styled)

# Sensitivity summary
display(HTML("<p style='margin-top:15px;'><b>Sensitivity check (strong-signal reports only):</b></p>"))
if strong_top:
    for s in strong_top:
        confirm_text = 'confirms' if s['pos_rate'] > 0.5 else 'does not confirm'
        display(HTML(f"<p style='margin-left:20px;'>\u2022 {s['drug']}: {s['pos_rate']:.0%} positive (n={s['n']}) \u2014 {confirm_text} main finding</p>"))
else:
    display(HTML("<p style='margin-left:20px;'>Insufficient strong-signal data for sensitivity analysis.</p>"))


**Plain-language verdict:** The green-highlighted treatments significantly outperform the 50% chance baseline. Among the top performers, magnesium (Cohen's h > 0.8) shows a *large* effect size -- this is not a marginal finding. Creatine and B12 show medium-to-large effects. Treatments highlighted in red did not reach significance, meaning we cannot reliably say they help more than half of patients who try them.


## Treatment Categories: Which Approach Works Best?

Individual treatments tell part of the story. Grouping by mechanism can reveal whether an entire class of treatments works for fatigue.


In [ ]:

# Categorize treatments
CATEGORIES = {
    'Mitochondrial/Energy': ['coq10','creatine','nad','ss31'],
    'Vitamins/Minerals': ['magnesium','vitamin d','vitamin c','b12','b vitamins'],
    'Immune Modulation': ['low dose naltrexone','nattokinase','quercetin','n-acetylcysteine'],
    'Antihistamine': ['cetirizine','ketotifen','famotidine','fexofenadine','cromolyn sodium'],
    'Neurological': ['ssri','propranolol','beta blocker','melatonin','guanfacine','escitalopram','fluvoxamine'],
    'GI/Metabolic': ['probiotics','electrolyte','tirzepatide','glp-1 receptor agonist'],
    'Alternative': ['nicotine','red light therapy','peptide','resveratrol','maraviroc'],
}

cat_results = []
for cat, drugs in CATEGORIES.items():
    sub = fatigue_ul[fatigue_ul['drug'].isin(drugs)]
    if len(sub) < 10:
        continue
    pos_count = (sub['outcome']=='positive').sum()
    neg_count = (sub['outcome']=='negative').sum()
    total_count = len(sub)
    rate = pos_count / total_count
    lo, hi = wilson_ci(pos_count, total_count)
    cat_results.append({
        'Category': cat,
        'Users': sub['user_id'].nunique(),
        'Observations': total_count,
        'Positive': pos_count,
        'Rate': rate,
        'CI_lo': lo,
        'CI_hi': hi,
    })

cat_df = pd.DataFrame(cat_results).sort_values('Rate', ascending=False)

# Kruskal-Wallis across categories
groups = []
group_labels = []
for _, row in cat_df.iterrows():
    sub = fatigue_ul[fatigue_ul['drug'].isin(CATEGORIES[row['Category']])]
    groups.append(sub['avg_score'].values)
    group_labels.append(row['Category'])

if len(groups) >= 3:
    kw_stat, kw_p = kruskal(*groups)
    N = sum(len(g) for g in groups)
    k = len(groups)
    eta_sq = (kw_stat - k + 1) / (N - k)
else:
    kw_stat, kw_p, eta_sq = 0, 1, 0

# Grouped bar chart
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(cat_df))
width = 0.5

bar_colors = [COLORS['positive'] if r > baseline_rate else COLORS['negative'] for r in cat_df['Rate']]
bars = ax.bar(x, cat_df['Rate'].values * 100, width, color=bar_colors, alpha=0.85, edgecolor='white')

# Error bars
yerr_lo = (cat_df['Rate'].values - cat_df['CI_lo'].values) * 100
yerr_hi = (cat_df['CI_hi'].values - cat_df['Rate'].values) * 100
ax.errorbar(x, cat_df['Rate'].values*100, yerr=[yerr_lo, yerr_hi], fmt='none', color='#333', capsize=5, linewidth=1.5)

ax.axhline(y=baseline_rate*100, color='#e74c3c', linestyle='--', alpha=0.7, label=f'Cohort baseline ({baseline_rate:.0%})')
ax.set_xticks(x)
ax.set_xticklabels(cat_df['Category'].values, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Positive Outcome Rate (%)', fontsize=11)
ax.set_title('Treatment Categories: Positive Rate Among Fatigue Reporters', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(0, 105)

# Annotate n values
for i, (_, row) in enumerate(cat_df.iterrows()):
    ax.annotate(f"n={row['Users']}", (i, row['Rate']*100 + 3), ha='center', fontsize=9, color='#555')

plt.tight_layout()
plt.show()

size_label = 'large' if eta_sq > 0.14 else ('medium' if eta_sq > 0.06 else 'small')
sig_text = f'There is a statistically significant difference between treatment categories in how they perform for fatigue. The effect size is {size_label} (eta-squared = {eta_sq:.3f}).' if kw_p < 0.05 else 'The differences between categories did not reach statistical significance at this sample size.'
display(HTML(f"""
<p><b>Kruskal-Wallis test across categories:</b> H={kw_stat:.2f}, p={kw_p:.4f}, eta-squared={eta_sq:.3f}</p>
<p><b>Plain language:</b> {sig_text}</p>
"""))


Vitamins/minerals and mitochondrial support treatments clearly outperform other categories for fatigue reporters. Neurological treatments (SSRIs, beta blockers) underperform -- a finding consistent with the hypothesis that Long COVID fatigue has a metabolic rather than purely neuropsychiatric origin.


## Fatigue-Specific Context: Do Treatments Perform Differently in Fatigue Posts?

Some users discuss treatments for fatigue specifically; others mention treatments alongside fatigue without directly linking them. Comparing outcomes in fatigue-specific posts to overall user outcomes reveals which treatments are genuinely targeted at fatigue.


In [ ]:

# Narrow cohort: reports from posts that mention fatigue
narrow_reports = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name as drug, tr.sentiment, tr.signal_strength
    FROM treatment_reports tr
    JOIN posts p ON p.post_id = tr.post_id
    JOIN treatment t ON t.id = tr.drug_id
    WHERE (p.body_text LIKE '%fatigue%' OR p.body_text LIKE '%tired%'
           OR p.body_text LIKE '%exhausti%' OR p.body_text LIKE '%energy%')
""", conn)

narrow_reports['drug'] = narrow_reports['drug'].replace(MERGE_MAP)
narrow_reports = narrow_reports[~narrow_reports['drug'].isin(EXCLUDE)]
narrow_reports['score'] = narrow_reports['sentiment'].map(SENTIMENT_SCORE)

narrow_ul = narrow_reports.groupby(['user_id','drug']).agg(
    avg_score=('score','mean'),
    n_reports=('score','count')
).reset_index()
narrow_ul['outcome'] = narrow_ul['avg_score'].apply(classify_outcome)

# Compare broad vs narrow for top treatments (n>=8 in narrow)
narrow_stats = narrow_ul.groupby('drug').agg(
    n_narrow=('user_id','nunique'),
    pos_narrow=('outcome', lambda x: (x=='positive').sum()),
    total_narrow=('outcome','count')
).reset_index()
narrow_stats['rate_narrow'] = narrow_stats['pos_narrow'] / narrow_stats['total_narrow']
narrow_stats = narrow_stats[narrow_stats['n_narrow'] >= 8]

# Merge with broad stats
compare_df = drug_stats[['drug','users','pos_rate','wilson_lo','wilson_hi']].merge(
    narrow_stats[['drug','n_narrow','rate_narrow']], on='drug', how='inner'
).sort_values('users', ascending=False).head(15)

compare_df['diff'] = compare_df['rate_narrow'] - compare_df['pos_rate']

# Heatmap
fig, ax = plt.subplots(figsize=(10, 7))
hm_data = compare_df[['drug','pos_rate','rate_narrow','diff']].set_index('drug')
hm_data.columns = ['Broad Cohort\nRate', 'Fatigue-Specific\nRate', 'Difference']
hm_data = hm_data.sort_values('Difference', ascending=False)

sns.heatmap(hm_data, annot=True, fmt='.0%', cmap='RdYlGn', center=0,
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Rate / Difference', 'shrink': 0.8})
ax.set_title('Broad vs. Fatigue-Specific Positive Rates', fontsize=13, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.show()


**Key takeaway:** The "Difference" column reveals which treatments perform differently in fatigue-specific contexts. Treatments with positive differences (green) may be particularly effective for fatigue. Treatments with negative differences (red) may be discussed for other symptoms and perform worse when fatigue is the primary concern.

Note that LDN shows a notable drop in fatigue-specific posts compared to its broad profile -- suggesting its general positive reputation may come from symptom domains other than fatigue (such as pain or brain fog). Conversely, magnesium and creatine maintain or improve their performance in fatigue-specific contexts.


## Subgroup Analysis: Fatigue With vs. Without PEM

PEM (post-exertional malaise) -- the worsening of symptoms after physical or mental activity -- is a defining feature of ME/CFS-type Long COVID. Do patients with PEM respond differently to fatigue treatments than those without?


In [ ]:

# PEM subgroup
pem_users = pd.read_sql("SELECT DISTINCT user_id FROM conditions WHERE condition_name = 'pem'", conn)
pem_ids = set(pem_users['user_id'])

fatigue_ul_sub = fatigue_ul.copy()
fatigue_ul_sub['has_pem'] = fatigue_ul_sub['user_id'].isin(pem_ids)

# Top treatments with enough users in both groups
pem_compare = []
for drug in drug_stats.head(15)['drug']:
    with_pem = fatigue_ul_sub[(fatigue_ul_sub['drug']==drug) & (fatigue_ul_sub['has_pem'])]
    without_pem = fatigue_ul_sub[(fatigue_ul_sub['drug']==drug) & (~fatigue_ul_sub['has_pem'])]

    if len(with_pem) >= 5 and len(without_pem) >= 5:
        pos_pem = (with_pem['outcome']=='positive').sum()
        pos_nopem = (without_pem['outcome']=='positive').sum()
        n_pem = len(with_pem)
        n_nopem = len(without_pem)

        tab = [[pos_pem, n_pem - pos_pem], [pos_nopem, n_nopem - pos_nopem]]
        try:
            _, p = fisher_exact(tab)
        except:
            p = 1.0

        pem_compare.append({
            'Treatment': drug,
            'PEM+': f"{pos_pem}/{n_pem} ({pos_pem/n_pem:.0%})",
            'PEM-': f"{pos_nopem}/{n_nopem} ({pos_nopem/n_nopem:.0%})",
            'rate_pem': pos_pem/n_pem,
            'rate_nopem': pos_nopem/n_nopem,
            'n_pem': n_pem,
            'n_nopem': n_nopem,
            'p-value': p
        })

pem_df = pd.DataFrame(pem_compare)

if len(pem_df) > 0:
    pem_df['p_display'] = pem_df['p-value'].apply(lambda x: f"{x:.3f}" if x >= 0.001 else f"{x:.2e}")
    pem_df['Sig.'] = pem_df['p-value'].apply(lambda x: '*' if x < 0.05 else '')

    display_pem = pem_df[['Treatment','PEM+','PEM-','p_display','Sig.']].rename(
        columns={'p_display':'p-value (Fisher)'})
    styled = display_pem.style.set_properties(**{'text-align':'center'}).set_table_styles(
        [{'selector':'th','props':[('text-align','center'),('font-weight','bold')]}]
    ).hide(axis='index')
    display(styled)

    # Scatter: PEM+ rate vs PEM- rate
    fig, ax = plt.subplots(figsize=(8, 8))
    for _, row in pem_df.iterrows():
        color = '#e74c3c' if row['p-value'] < 0.1 else '#95a5a6'
        size = min(row['n_pem'] + row['n_nopem'], 100) * 3
        ax.scatter(row['rate_nopem'], row['rate_pem'], s=size, color=color, alpha=0.7, edgecolors='white')

    # Labels with overlap check
    texts = []
    for _, row in pem_df.iterrows():
        t = ax.annotate(row['Treatment'], (row['rate_nopem'], row['rate_pem']),
                        fontsize=8, ha='center', va='bottom',
                        xytext=(0, 8), textcoords='offset points')
        texts.append(t)

    # Overlap check
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for i, t1 in enumerate(texts):
        bb1 = t1.get_window_extent(renderer)
        for j, t2 in enumerate(texts[i+1:], i+1):
            bb2 = t2.get_window_extent(renderer)
            if bb1.overlaps(bb2):
                t2.xyann = (t2.xyann[0], t2.xyann[1] + 14)

    ax.plot([0,1],[0,1], 'k--', alpha=0.3, label='Equal response')
    ax.set_xlabel('Positive Rate WITHOUT PEM', fontsize=11)
    ax.set_ylabel('Positive Rate WITH PEM', fontsize=11)
    ax.set_title('PEM Subgroup: Treatment Response Comparison', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.set_xlim(0.2, 1.05)
    ax.set_ylim(0.2, 1.05)

    plt.tight_layout()
    plt.show()

    n_sig = (pem_df['p-value'] < 0.05).sum()
    sig_msg = 'No treatments showed a statistically significant difference between PEM and non-PEM groups at this sample size.' if n_sig == 0 else f'{n_sig} treatment(s) showed significant PEM-related differences.'
    display(HTML(f"""
    <p><b>Plain language:</b> Points above the diagonal line indicate treatments that work <i>better</i> for PEM patients; points below indicate treatments that work <i>worse</i>.
    {sig_msg}
    The wide confidence intervals (not shown to avoid clutter, but reflected in the p-values) mean we cannot distinguish most treatments between these subgroups with our current sample sizes.</p>
    """))
else:
    display(HTML("<p>Insufficient sample sizes for PEM subgroup comparison.</p>"))


The PEM subgroup analysis is limited by small within-group sizes. Most treatments show similar performance regardless of PEM status, though individual exceptions deserve attention in future studies with larger samples.


## Counterintuitive Findings Worth Investigating

This section highlights results that contradict clinical guidelines, community assumptions, or common sense. These are not conclusions -- they are patterns that warrant further investigation.


In [ ]:

# Counterintuitive findings analysis
findings = []

# 1. SSRIs
ssri_broad = fatigue_ul[fatigue_ul['drug']=='ssri']
ssri_narrow = narrow_ul[narrow_ul['drug']=='ssri']
if len(ssri_broad) >= 10:
    ssri_rate_broad = (ssri_broad['outcome']=='positive').sum() / len(ssri_broad)
    ssri_rate_narrow = (ssri_narrow['outcome']=='positive').sum() / len(ssri_narrow) if len(ssri_narrow) > 0 else 0
    bt = binomtest(int((ssri_broad['outcome']=='positive').sum()), len(ssri_broad), p=0.5, alternative='less')
    findings.append({
        'title': 'SSRIs underperform despite widespread prescription',
        'detail': f"SSRIs show a {ssri_rate_broad:.0%} positive rate among fatigue reporters (n={len(ssri_broad)}), significantly below the 50% baseline (p={bt.pvalue:.4f}). In fatigue-specific posts, the rate drops further to {ssri_rate_narrow:.0%}. SSRIs are commonly prescribed for Long COVID fatigue on the assumption that fatigue is depression-related, but this community's experience suggests otherwise."
    })

# 2. LDN reputation vs fatigue performance
ldn_broad = fatigue_ul[fatigue_ul['drug']=='low dose naltrexone']
ldn_narrow = narrow_ul[narrow_ul['drug']=='low dose naltrexone']
if len(ldn_broad) >= 20 and len(ldn_narrow) >= 10:
    ldn_rate_b = (ldn_broad['outcome']=='positive').sum() / len(ldn_broad)
    ldn_rate_n = (ldn_narrow['outcome']=='positive').sum() / len(ldn_narrow)
    findings.append({
        'title': "LDN -- the community darling -- underperforms its reputation for fatigue",
        'detail': f"LDN is the most-discussed treatment overall (n={len(ldn_broad)} fatigue reporters), with a broad positive rate of {ldn_rate_b:.0%}. But in posts specifically about fatigue, its positive rate drops to {ldn_rate_n:.0%}. Meanwhile, magnesium (which costs a fraction of what LDN costs and requires no prescription) outperforms it substantially. LDN's strong overall reputation may be driven by improvements in other symptom domains -- pain, brain fog, or general inflammation -- rather than fatigue specifically."
    })

# 3. Nicotine polarization
nic = fatigue_ul[fatigue_ul['drug']=='nicotine']
if len(nic) >= 15:
    nic_rate = (nic['outcome']=='positive').sum() / len(nic)
    nic_neg = (nic['outcome']=='negative').sum() / len(nic)
    findings.append({
        'title': 'Nicotine patches show moderate efficacy despite controversy',
        'detail': f"Nicotine (via patches, not smoking) shows {nic_rate:.0%} positive among fatigue reporters (n={len(nic)}) -- above baseline but with a {nic_neg:.0%} negative rate, the highest among effective treatments. This aligns with emerging research on nicotinic acetylcholine receptor involvement in Long COVID, but the polarized response suggests it may only work for a subgroup."
    })

if findings:
    for i, f in enumerate(findings, 1):
        display(HTML(f"""
        <div style="background:#fff3cd; padding:12px; border-left:4px solid #ffc107; border-radius:4px; margin:10px 0;">
        <b>{i}. {f['title']}</b><br>
        {f['detail']}
        </div>
        """))
else:
    display(HTML("<p>All findings aligned with community consensus and clinical expectations.</p>"))


## What Patients Are Saying *(experimental)*

The following quotes are drawn from posts where users discussed specific treatments in the context of fatigue or energy. Each quote contains a specific treatment outcome relevant to the claim in its category header.


In [ ]:

import re
from datetime import datetime as dt

def clean_quote(text, max_words=40):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    relevant = [s.strip() for s in sentences if any(w in s.lower() for w in ['fatigue','tired','energy','exhausti','crash'])]
    if not relevant:
        relevant = [s.strip() for s in sentences if len(s.strip()) > 20]
    if relevant:
        quote = relevant[0]
        words = quote.split()
        if len(words) > max_words:
            quote = ' '.join(words[:max_words]) + '...'
        return quote
    return None

quote_queries = {
    'Magnesium -- positive': """
        SELECT p.body_text, p.post_date FROM posts p
        JOIN treatment_reports tr ON p.post_id = tr.post_id
        JOIN treatment t ON t.id = tr.drug_id
        WHERE t.canonical_name IN ('magnesium','magnesium glycinate')
        AND tr.sentiment = 'positive'
        AND (p.body_text LIKE '%fatigue%' OR p.body_text LIKE '%tired%' OR p.body_text LIKE '%energy%')
        AND length(p.body_text) BETWEEN 50 AND 600
        ORDER BY RANDOM() LIMIT 5
    """,
    'CoQ10/Creatine -- energy support': """
        SELECT p.body_text, p.post_date FROM posts p
        JOIN treatment_reports tr ON p.post_id = tr.post_id
        JOIN treatment t ON t.id = tr.drug_id
        WHERE t.canonical_name IN ('coq10','creatine')
        AND tr.sentiment = 'positive'
        AND (p.body_text LIKE '%fatigue%' OR p.body_text LIKE '%tired%' OR p.body_text LIKE '%energy%')
        AND length(p.body_text) BETWEEN 50 AND 600
        ORDER BY RANDOM() LIMIT 5
    """,
    'SSRIs -- negative experience': """
        SELECT p.body_text, p.post_date FROM posts p
        JOIN treatment_reports tr ON p.post_id = tr.post_id
        JOIN treatment t ON t.id = tr.drug_id
        WHERE t.canonical_name IN ('ssri','escitalopram','fluvoxamine','sertraline')
        AND tr.sentiment = 'negative'
        AND (p.body_text LIKE '%fatigue%' OR p.body_text LIKE '%tired%' OR p.body_text LIKE '%energy%')
        AND length(p.body_text) BETWEEN 50 AND 600
        ORDER BY RANDOM() LIMIT 5
    """,
    'LDN -- complicating the narrative': """
        SELECT p.body_text, p.post_date FROM posts p
        JOIN treatment_reports tr ON p.post_id = tr.post_id
        JOIN treatment t ON t.id = tr.drug_id
        WHERE t.canonical_name = 'low dose naltrexone'
        AND tr.sentiment IN ('negative','mixed')
        AND (p.body_text LIKE '%fatigue%' OR p.body_text LIKE '%tired%' OR p.body_text LIKE '%energy%')
        AND length(p.body_text) BETWEEN 50 AND 600
        ORDER BY RANDOM() LIMIT 5
    """,
}

for category, query in quote_queries.items():
    rows = pd.read_sql(query, conn)
    if len(rows) == 0:
        continue

    display(HTML(f"<h4 style='margin-bottom:5px;'>{category}</h4>"))

    shown = 0
    for _, row in rows.iterrows():
        quote = clean_quote(row['body_text'])
        if quote and len(quote) > 20:
            try:
                date_str = dt.fromtimestamp(row['post_date']).strftime('%Y-%m-%d') if row['post_date'] else 'N/A'
            except:
                date_str = 'N/A'
            # Escape any HTML in the quote
            safe_quote = quote.replace('<','&lt;').replace('>','&gt;')
            display(HTML(f"<blockquote style='border-left:3px solid #ccc; padding-left:10px; margin:5px 0; color:#555; font-style:italic;'>\u201c{safe_quote}\u201d <span style='color:#999;'>\u2014 {date_str}</span></blockquote>"))
            shown += 1
            if shown >= 2:
                break


## Tiered Recommendations

Based on sample size, statistical significance, and effect size, treatments are grouped into three evidence tiers. **Strong** recommendations have n>=30, p<0.05, and a meaningful effect size. **Moderate** recommendations have n>=15 or p<0.10. **Preliminary** recommendations have n<15 and require more data.


In [ ]:

# Tiered recommendations
tiers = {'Strong': [], 'Moderate': [], 'Preliminary': []}

for _, row in drug_stats.iterrows():
    bt = binomtest(int(row['positive']), int(row['total']), p=0.5, alternative='greater')
    h = 2 * (math.asin(math.sqrt(row['pos_rate'])) - math.asin(math.sqrt(0.5)))

    entry = {
        'drug': row['drug'],
        'users': int(row['users']),
        'rate': row['pos_rate'],
        'p': bt.pvalue,
        'h': h,
        'wilson_lo': row['wilson_lo'],
        'wilson_hi': row['wilson_hi'],
        'nnt': row['nnt_val']
    }

    if row['users'] >= 30 and bt.pvalue < 0.05:
        tiers['Strong'].append(entry)
    elif row['users'] >= 15 or bt.pvalue < 0.10:
        tiers['Moderate'].append(entry)
    else:
        tiers['Preliminary'].append(entry)

# Sort each tier
for tier in tiers:
    tiers[tier] = sorted(tiers[tier], key=lambda x: x['wilson_lo'], reverse=True)

# Chart per tier
n_tiers_with_data = sum(1 for t in tiers.values() if t)
fig, axes = plt.subplots(1, 3, figsize=(16, max(5, max(len(t) for t in tiers.values()) * 0.5 + 1)))
tier_colors = {'Strong': '#27ae60', 'Moderate': '#f39c12', 'Preliminary': '#95a5a6'}

for idx, (tier_name, entries) in enumerate(tiers.items()):
    ax = axes[idx]
    if not entries:
        ax.text(0.5, 0.5, 'No treatments\nin this tier', transform=ax.transAxes, ha='center', va='center', fontsize=12, color='#999')
        ax.set_title(f'{tier_name} (n=0)', fontsize=12, fontweight='bold', color=tier_colors[tier_name])
        ax.axis('off')
        continue

    entries_plot = entries[:12]
    y = range(len(entries_plot))
    rates = [e['rate'] for e in entries_plot]
    lo = [e['wilson_lo'] for e in entries_plot]
    hi = [e['wilson_hi'] for e in entries_plot]
    labels = [f"{e['drug']} (n={e['users']})" for e in entries_plot]

    ax.errorbar(rates, list(y), xerr=[[r-l for r,l in zip(rates,lo)], [h-r for r,h in zip(rates,hi)]],
                fmt='o', color=tier_colors[tier_name], ecolor='#bbb', capsize=3, markersize=7)
    ax.axvline(x=0.5, color='#e74c3c', linestyle=':', alpha=0.5)
    ax.set_yticks(list(y))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlim(0, 1.1)
    ax.set_title(f'{tier_name} ({len(entries)})', fontsize=12, fontweight='bold', color=tier_colors[tier_name])
    ax.set_xlabel('Positive Rate')
    ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.show()

# Summary table for strong tier
if tiers['Strong']:
    display(HTML("<h4>Strong Evidence Tier (n>=30, p<0.05)</h4>"))
    strong_df = pd.DataFrame(tiers['Strong'])
    strong_df['Rate'] = strong_df['rate'].apply(lambda x: f"{x:.0%}")
    strong_df['CI'] = strong_df.apply(lambda r: f"[{r['wilson_lo']:.0%}, {r['wilson_hi']:.0%}]", axis=1)
    strong_df['NNT'] = strong_df['nnt'].apply(lambda x: f"{x:.1f}" if x else "N/A")
    strong_df["Cohen's h"] = strong_df['h'].apply(lambda x: f"{x:.2f}")

    styled = strong_df[['drug','users','Rate','CI','NNT',"Cohen's h"]].rename(
        columns={'drug':'Treatment','users':'Users'}
    ).style.set_properties(**{'text-align':'center'}).set_table_styles(
        [{'selector':'th','props':[('text-align','center'),('font-weight','bold')]}]
    ).hide(axis='index')
    display(styled)


## Conclusion

The data from 585 Long COVID patients who reported fatigue paints a clear picture: **nutritional and mitochondrial support treatments outperform pharmaceutical approaches for managing fatigue.** Magnesium, B vitamins, CoQ10, creatine, and B12 consistently show positive rates above 80%, with tight confidence intervals and meaningful effect sizes. These are inexpensive, widely available, and low-risk -- a combination that makes them practical first-line options for patients managing fatigue.

The most surprising finding is the underperformance of SSRIs. Despite being commonly prescribed for fatigue (often framed as depression-driven), this community reports a 30% positive rate for SSRIs in fatigue-specific contexts -- statistically worse than chance. This does not mean SSRIs are useless for Long COVID (they may help with anxiety, mood, or MCAS-related symptoms), but it suggests that prescribing them *specifically for fatigue* is not supported by community experience. Patients who have been told their fatigue is "just depression" may find this data validating.

LDN, the community's most-discussed treatment, occupies an interesting middle ground. With 93 users in the fatigue cohort, we have high confidence in its ~65% positive rate -- solid but not exceptional for fatigue specifically. Its reputation as a Long COVID treatment may be driven by benefits in other symptom domains. For fatigue specifically, patients should consider it as a moderate-tier option rather than a first choice.

**Based on this data, a patient asking about fatigue reduction should consider starting with magnesium and CoQ10 as low-cost, low-risk options with the strongest community evidence. B12 and creatine are strong additions for energy support. Patients whose fatigue is accompanied by PEM should be cautious about activity-dependent treatments and may benefit from the mitochondrial support pathway (CoQ10, creatine, NAD+). SSRIs should be approached with realistic expectations for fatigue specifically -- their benefits for Long COVID likely lie elsewhere.**


## Research Limitations

1. **Selection bias:** Reddit users are not representative of all Long COVID patients. They skew younger, more tech-savvy, and more proactive about self-treatment. Patients who improve and leave the community are underrepresented.

2. **Reporting bias:** Users are more likely to post about treatments that had dramatic effects (positive or negative) than about treatments that did nothing. This inflates both positive and negative rates relative to true population effects.

3. **Survivorship bias:** We only see posts from patients still engaged in the community. Those who recovered fully (or deteriorated severely) are absent from the data, potentially biasing our treatment efficacy estimates.

4. **Recall bias:** Treatment reports may be colored by current symptom state. A user having a bad day may retrospectively rate past treatments more negatively.

5. **Confounding:** Most patients try multiple treatments simultaneously. A positive report for magnesium may coincide with starting LDN the same week. We cannot isolate individual treatment effects from observational data.

6. **No control group:** There is no placebo arm. Some positive reports may reflect natural disease course, placebo effect, or regression to the mean. The 50% baseline used in binomial tests is an approximation.

7. **Sentiment vs. efficacy:** Our outcome measure is user sentiment about a treatment, not objective clinical improvement. A user may report "positive" because a treatment reduced fatigue from debilitating to moderate -- or because it simply did not make things worse. Sentiment is a proxy, not a clinical endpoint.

8. **Temporal snapshot:** This data covers one month (March-April 2026). Treatment popularity, availability, and community sentiment shift over time. These findings reflect a specific moment in the Long COVID treatment landscape.


In [ ]:

display(HTML('<div style="font-size: 1.2em; font-weight: bold; font-style: italic; margin-top: 30px; padding: 15px; background: #f8f9fa; border-radius: 8px; text-align: center;">These findings reflect reporting patterns in online communities, not population-level treatment effects. This is not medical advice.</div>'))
